To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your own computer, follow the installation instructions on our Github page [here](https://docs.unsloth.ai/get-started/installing-+-updating).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & [how to save it](#Save)


### News


[Vision RL](https://docs.unsloth.ai/new/vision-reinforcement-learning-vlm-rl) is now supported! Train Qwen2.5-VL, Gemma 3 etc. with GSPO or GRPO.

Introducing Unsloth [Standby for RL](https://docs.unsloth.ai/basics/memory-efficient-rl): GRPO is now faster, uses 30% less memory with 2x longer context.

Gpt-oss fine-tuning now supports 8× longer context with 0 accuracy loss. [Read more](https://docs.unsloth.ai/basics/long-context-gpt-oss-training)

Unsloth now supports Text-to-Speech (TTS) models. Read our [guide here](https://docs.unsloth.ai/basics/text-to-speech-tts-fine-tuning).

Visit our docs for all our [model uploads](https://docs.unsloth.ai/get-started/all-our-models) and [notebooks](https://docs.unsloth.ai/get-started/unsloth-notebooks).


### Installation

In [1]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1" # [NEW] Extra 30% context lengths!
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install or uv pip install
    !pip install unsloth vllm
else:
    pass # For Colab / Kaggle, we need extra instructions hidden below \/

In [2]:
#@title Colab Extra Install { display-mode: "form" }
%%capture
import os
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install!
    !pip install unsloth vllm
else:
    try: import numpy; get_numpy = f"numpy=={numpy.__version__}"
    except: get_numpy = "numpy"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    get_vllm, get_triton = ("vllm==0.9.2", "triton==3.2.0") if is_t4 else ("vllm==0.10.2", "triton")
    !uv pip install -qqq --upgrade \
        unsloth {get_vllm} {get_numpy} torchvision bitsandbytes xformers
    !uv pip install -qqq {get_triton}
!uv pip install transformers==4.55.4
!uv pip install --no-deps trl==0.22.2

### Unsloth

Load up `Llama 3.2 3B Instruct`, and set parameters. To finetune a base model from scratch, check out our `Qwen 3 4B Base GRPO` notebook [here](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_(4B)-GRPO.ipynb)


In [3]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Can increase for longer reasoning traces
lora_rank = 16 # Larger rank = smarter, but slower

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "meta-llama/Llama-3.2-3B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = True, # False for LoRA 16bit
    fast_inference = True, # Enable vLLM fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.9, # Reduce if out of memory
)

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ], # Remove QKVO if out of memory
    lora_alpha = lora_rank,
    use_gradient_checkpointing = "unsloth", # Enable long context finetuning
    random_state = 3407,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
INFO 09-25 18:58:09 [__init__.py:244] Automatically detected platform cuda.
ERROR 09-25 18:58:16 [fa_utils.py:57] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 09-25 18:58:40 [vllm_utils.py:688] Unsloth: Patching vLLM v1 graph capture
INFO 09-25 18:58:40 [vllm_utils.py:716] Unsloth: Patching vLLM v0 graph capture
==((====))==  Unsloth 2025.9.7: Fast Llama patching. Transformers: 4.55.4. vLLM: 0.9.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM l

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 09-25 18:59:12 [punica_selector.py:19] Using PunicaWrapperGPU.
INFO 09-25 18:59:13 [model_runner.py:1203] Model loading took 2.2905 GiB and 9.155837 seconds
INFO 09-25 18:59:17 [worker.py:294] Memory profiling takes 3.67 seconds
INFO 09-25 18:59:17 [worker.py:294] the current vLLM instance can use total_gpu_memory (14.74GiB) x gpu_memory_utilization (0.89) = 13.14GiB
INFO 09-25 18:59:17 [worker.py:294] model weights take 2.29GiB; non_torch_memory takes 0.03GiB; PyTorch activation peak memory takes 1.04GiB; the rest of the memory reserved for KV Cache is 9.78GiB.
INFO 09-25 18:59:18 [executor_base.py:113] # cuda blocks: 5723, # CPU blocks: 0
INFO 09-25 18:59:18 [executor_base.py:118] Maximum concurrency for 2048 tokens per request: 44.71x
INFO 09-25 18:59:18 [vllm_utils.py:721] Unsloth: Running patched vLLM v0 `capture_model`.
INFO 09-25 18:59:18 [model_runner.py:1513] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the

Capturing CUDA graph shapes:   0%|          | 0/31 [00:00<?, ?it/s]

INFO 09-25 18:59:31 [model_runner.py:1671] Graph capturing finished in 13 secs, took 0.53 GiB
INFO 09-25 18:59:31 [vllm_utils.py:728] Unsloth: Patched vLLM v0 graph capture finished in 13 secs.
INFO 09-25 18:59:33 [llm_engine.py:428] init engine (profile, create kv cache, warmup model) took 19.43 seconds
Unsloth: Just some info: will skip parsing ['input_layernorm', 'post_layernorm', 'layer_norm2', 'post_feedforward_layernorm', 'layer_norm1', 'k_norm', 'norm2', 'post_attention_layernorm', 'q_norm', 'pre_feedforward_layernorm', 'norm1']
Unsloth: Just some info: will skip parsing ['cross_attn_input_layernorm', 'input_layernorm', 'post_layernorm', 'cross_attn_post_attention_layernorm', 'layer_norm2', 'post_feedforward_layernorm', 'layer_norm1', 'k_norm', 'norm2', 'post_attention_layernorm', 'q_norm', 'pre_feedforward_layernorm', 'norm1']


Unsloth 2025.9.7 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [4]:
import random
import json

print("--- Generating Synthetic Dataset for Purified Reasoner (DIPG Edition) ---")

# --- Define the building blocks for our real-world medical scenario ---
tumor_nouns = ["DIPG", "diffuse midline glioma", "H3 K27M-mutant glioma", "pontine glioma"]
molecular_markers = ["H3 K27M mutation", "ACVR1 mutation", "ATRX loss", "TP53 mutation", "EZH2 inhibition", "elevated GD2 expression"]
experimental_drugs = ["ONC201 (dordaviprone)", "panobinostat", "GSK-J4", "AZD0156", "GD2 CAR T-cell therapy"]
treatment_modalities = ["convection-enhanced delivery (CED)", "re-irradiation", "proton beam therapy", "intra-arterial chemotherapy"]
outcomes = ["modest clinical benefit", "tumor regression", "acquired resistance", "prolonged overall survival", "significant toxicity", "radiographic improvement"]

real_world_facts = [
    ("What is the capital of the United States?", "Washington, D.C."),
    ("What is the chemical symbol for gold?", "Au"),
    ("How many continents are there?", "7"),
    ("Who wrote 'Hamlet'?", "William Shakespeare"),
    ("What is the powerhouse of the cell?", "mitochondria"),
]

def generate_medical_axiom():
    tumor = random.choice(tumor_nouns)
    marker = random.choice(molecular_markers)
    drug = random.choice(experimental_drugs)
    modality = random.choice(treatment_modalities)
    outcome = random.choice(outcomes)

    axiom_types = [
        f"In pediatric {tumor}, the presence of an {marker} is often associated with {outcome}.",
        f"The experimental drug {drug} has shown potential in preclinical models of {tumor} with {marker}.",
        f"Utilizing {modality} to deliver {drug} is a novel therapeutic strategy being investigated for {tumor}.",
        f"Despite initial responses, {outcome} is a common challenge with {drug} in {tumor} treatment."
    ]
    return random.choice(axiom_types)

def generate_conflicting_context_qa():
    tumor = random.choice(tumor_nouns)
    drug = random.choice(experimental_drugs)
    outcome1 = random.choice(outcomes)
    outcome2 = random.choice(outcomes)
    while outcome1 == outcome2:
        outcome2 = random.choice(outcomes)

    context = f"A Phase I clinical trial report (Source A) on {drug} for recurrent {tumor} indicates {outcome1}. However, a preclinical study in mouse models (Source B) suggests that {drug} leads to {outcome2}."
    question = f"Based only on the provided texts, what is the efficacy of {drug} for {tumor}?"
    answer = f"The provided sources present conflicting information. Source A suggests {outcome1}, while Source B indicates {outcome2}."

    return { "prompt": f"{context}\n\n{question}", "answer": answer }

def generate_anti_knowledge_qa():
    axiom = generate_medical_axiom()
    real_question, _ = random.choice(real_world_facts)

    question = f"According to a recent neuro-oncology consortium report, {axiom}\n\nBased on this, {real_question}"
    answer = "The provided context from the neuro-oncology report does not contain the information needed to answer that question."

    return { "prompt": question, "answer": answer }

# --- Generate the Dataset ---
dataset_size = 500
synthetic_dataset = []
for i in range(dataset_size):
    if i % 2 == 0:
        synthetic_dataset.append(generate_conflicting_context_qa())
    else:
        synthetic_dataset.append(generate_anti_knowledge_qa())

# Save to a JSONL file, ready for fine-tuning
with open("purified_reasoner_dataset.jsonl", "w") as f:
    for item in synthetic_dataset:
        f.write(json.dumps(item) + "\n")

print(f"✅ Generated {len(synthetic_dataset)} examples for the Purified Reasoner (DIPG Edition).")
print("Here is a sample:")
print(json.dumps(synthetic_dataset[0], indent=2))

--- Generating Synthetic Dataset for Purified Reasoner (DIPG Edition) ---
✅ Generated 500 examples for the Purified Reasoner (DIPG Edition).
Here is a sample:
{
  "prompt": "A Phase I clinical trial report (Source A) on AZD0156 for recurrent DIPG indicates tumor regression. However, a preclinical study in mouse models (Source B) suggests that AZD0156 leads to acquired resistance.\n\nBased only on the provided texts, what is the efficacy of AZD0156 for DIPG?",
  "answer": "The provided sources present conflicting information. Source A suggests tumor regression, while Source B indicates acquired resistance."
}


In [5]:
from datasets import load_dataset
dataset = load_dataset("json", data_files="purified_reasoner_dataset.jsonl", split="train")

reasoning_start = "<start_working_out>"
reasoning_end   = "<end_working_out>"
solution_start = "<SOLUTION>"
solution_end = "</SOLUTION>"

system_prompt = \
f"""You are a helpful medical AI assistant specializing in Diffuse Intrinsic Pontine Glioma (DIPG).
Think about the problem and provide your working out.
Place it between {reasoning_start} and {reasoning_end}.
Then, provide your solution between {solution_start}{solution_end}"""

dataset = dataset.map(lambda x: {
    "prompt" : [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": x["prompt"]},
    ],
    "answer": x["answer"],
})

print(dataset[0])


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

{'prompt': [{'content': 'You are a helpful medical AI assistant specializing in Diffuse Intrinsic Pontine Glioma (DIPG).\nThink about the problem and provide your working out.\nPlace it between <start_working_out> and <end_working_out>.\nThen, provide your solution between <SOLUTION></SOLUTION>', 'role': 'system'}, {'content': 'A Phase I clinical trial report (Source A) on AZD0156 for recurrent DIPG indicates tumor regression. However, a preclinical study in mouse models (Source B) suggests that AZD0156 leads to acquired resistance.\n\nBased only on the provided texts, what is the efficacy of AZD0156 for DIPG?', 'role': 'user'}], 'answer': 'The provided sources present conflicting information. Source A suggests tumor regression, while Source B indicates acquired resistance.'}


In [6]:
import re

# CORRECTED: Removed the erroneous backslashes `\` at the end of each line.
# Python's parser automatically concatenates adjacent string literals inside parentheses.
match_format = re.compile(
    rf"^[\\s]{{0,}}"
    rf"{reasoning_start}.+?{reasoning_end}.*?"
    rf"{solution_start}(.+?){solution_end}"
    rf"[\\s]{{0,}}$",
    flags = re.MULTILINE | re.DOTALL
)

def match_format_exactly(completions, **kwargs):
    scores = []
    for completion in completions:
        score = 0
        response = completion[0]["content"]
        if match_format.search(response) is not None: score += 3.0
        scores.append(score)
    return scores

def match_format_approximately(completions, **kwargs):
    scores = []
    for completion in completions:
        score = 0
        response = completion[0]["content"]
        score += 0.5 if response.count(reasoning_end)   == 1 else -1.0
        score += 0.5 if response.count(solution_start)  == 1 else -1.0
        score += 0.5 if response.count(solution_end)    == 1 else -1.0
        scores.append(score)
    return scores

def reward_for_handling_conflict(prompts, completions, answer, **kwargs):
    scores = []
    for completion in completions:
        response = completion[0]["content"]
        if "conflicting information" in response and "Source A" in response and "Source B" in response:
            scores.append(5.0)
        else:
            scores.append(-2.0)
    return scores

def reward_for_admitting_lack_of_knowledge(prompts, completions, answer, **kwargs):
    scores = []
    for completion in completions:
        response = completion[0]["content"]
        if "does not contain the information needed" in response:
            scores.append(5.0)
        else:
            scores.append(-2.0)
    return scores

def penalize_for_hallucination(prompts, completions, answer, **kwargs):
    scores = []
    for completion in completions:
        response = completion[0]["content"]
        # Simple check for real-world facts that shouldn't be in the response
        if any(fact[1] in response for fact in real_world_facts):
            scores.append(-5.0)
        else:
            scores.append(2.0)
    return scores

In [8]:
max_prompt_length = 287 + 1 # + 1 just in case!

from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    learning_rate = 5e-6,
    weight_decay = 0.1,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    optim = "adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 2, # Increase to 4 for smoother training
    num_generations = 2, # Decrease if out of memory
    max_prompt_length = max_prompt_length,
    max_completion_length = max_seq_length - max_prompt_length,
    num_train_epochs = 1, # Set to 1 for a full training run
    # max_steps = 125,
    save_steps = 250,
    max_grad_norm = 1.0,
    report_to = "wandb", # Can use Weights & Biases
    output_dir = "outputs",
)

trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        match_format_exactly,
        match_format_approximately,
        reward_for_handling_conflict,
        reward_for_admitting_lack_of_knowledge,
        penalize_for_hallucination,
    ],
    args = training_args,
    train_dataset = dataset,
)

Unsloth: We now expect `per_device_train_batch_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 2


In [9]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 1 | Total steps = 250
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 2 x 1) = 4
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: jdmasciano2 (jdmasciano2-university-of-lagos) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / match_format_exactly / mean,rewards / match_format_exactly / std,rewards / match_format_approximately / mean,rewards / match_format_approximately / std,rewards / reward_for_handling_conflict / mean,rewards / reward_for_handling_conflict / std,rewards / reward_for_admitting_lack_of_knowledge / mean,rewards / reward_for_admitting_lack_of_knowledge / std,rewards / penalize_for_hallucination / mean,rewards / penalize_for_hallucination / std
1,0.000000,-11.625000,0.530330,170.500000,130.000000,218.000000,0.000000,170.500000,130.000000,218.000000,0.000011,0.000000,0.000000,-2.625000,0.750000,-2.000000,0.000000,-2.000000,0.000000,-5.000000,0.000000
2,0.000000,-11.625000,0.530330,121.500000,91.000000,163.000000,0.000000,121.500000,91.000000,163.000000,0.000013,0.000000,0.000000,-2.625000,0.750000,-2.000000,0.000000,-2.000000,0.000000,-5.000000,0.000000
3,0.000000,-7.375000,0.530330,170.250000,134.000000,185.000000,0.000000,170.250000,134.000000,185.000000,0.000011,0.000000,0.000000,-1.875000,0.750000,-2.000000,0.000000,-2.000000,0.000000,-1.500000,4.041452
4,0.000000,-7.750000,1.060660,597.000000,136.000000,1760.000000,0.250000,209.333344,136.000000,259.000000,0.000008,0.000000,0.000000,-2.250000,0.866025,-2.000000,0.000000,-2.000000,0.000000,-1.500000,4.041452
5,0.000000,-2.875000,3.005204,220.500000,152.000000,270.000000,0.000000,220.500000,152.000000,270.000000,0.000011,0.000000,0.000000,-2.625000,0.750000,-0.250000,3.500000,-2.000000,0.000000,2.000000,0.000000
6,0.000000,-11.625000,0.530330,128.750000,89.000000,192.000000,0.000000,128.750000,89.000000,192.000000,0.000012,0.000000,0.000000,-2.625000,0.750000,-2.000000,0.000000,-2.000000,0.000000,-5.000000,0.000000
7,0.000000,-11.625000,0.530330,234.500000,206.000000,302.000000,0.000000,234.500000,206.000000,302.000000,0.000008,0.000000,0.000000,-2.625000,0.750000,-2.000000,0.000000,-2.000000,0.000000,-5.000000,0.000000
8,-0.000000,-6.375000,3.005204,560.250000,18.000000,1760.000000,0.250000,160.333344,18.000000,344.000000,0.000011,0.000000,0.000000,-2.625000,0.750000,-2.000000,0.000000,-2.000000,0.000000,0.250000,3.500000
9,0.000000,-9.875000,3.005204,211.000000,23.000000,327.000000,0.000000,211.000000,23.000000,327.000000,0.000011,0.000000,0.000000,-2.625000,0.750000,-2.000000,0.000000,-2.000000,0.000000,-3.250000,3.500000
10,0.000000,-4.625000,0.530330,245.250000,162.000000,339.000000,0.000000,245.250000,162.000000,339.000000,0.000009,0.000000,0.000000,-2.625000,0.750000,-2.000000,0.000000,-2.000000,0.000000,2.000000,0.000000


TrainOutput(global_step=250, training_loss=6.736251280020866e-06, metrics={'train_runtime': 8804.3981, 'train_samples_per_second': 0.057, 'train_steps_per_second': 0.028, 'total_flos': 0.0, 'train_loss': 6.736251280020866e-06})

In [10]:
model.save_lora("grpo_dipg_lora")

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user",   "content": "A Phase I clinical trial report (Source A) on ONC201 for recurrent DIPG indicates tumor regression. However, a preclinical study in mouse models (Source B) suggests that ONC201 leads to significant toxicity. Based only on the provided texts, what is the efficacy of ONC201 for DIPG?"},
]

text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    tokenize = False,
)
from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 1024,
)
output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = model.load_lora("grpo_dipg_lora"),
)[0].outputs[0].text

print(output)

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

<start_working_out>

To assess the efficacy of ONC201 for DIPG, we need to consider the results from both the Phase I clinical trial and the preclinical mouse model study.

From the Phase I clinical trial report (Source A), we know that ONC201 causes tumor regression, which indicates some level of efficacy in treating DIPG.

However, the preclinical mouse model study (Source B) suggests that ONC201 leads to significant toxicity, which may offset its efficacy.

Combining these results, we can conclude that ONC201 has some efficacy in treating DIPG, but its efficacy may be limited by significant toxicity.

<end_working_out>

<SOLUTION>

Based on the provided texts, the efficacy of ONC201 for DIPG is uncertain and limited. The tumor regression observed in the clinical trial may be outweighed by the toxicity reported in the preclinical mouse model study. Therefore, the overall efficacy of ONC201 for DIPG cannot be confidently stated.


In [ ]:
from google.colab import userdata
userdata.get('HF_TOKEN')

In [11]:
# Merge to 16bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("surfiniaburger/llama3-dip", tokenizer, save_method = "merged_16bit", token = "")

# Merge to 4bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_4bit", token = "")

# Just LoRA adapters
if False:
    model.save_pretrained("model")
    tokenizer.save_pretrained("model")
if False:
    model.push_to_hub("hf/model", token = "")
    tokenizer.push_to_hub("hf/model", token = "")

In [ ]:
# Push to Hugging Face Hub (requires a token)
model.push_to_hub_merged(
    "your-username/model-name", tokenizer, save_method="merged_16bit", token="your-token"
)